# SYNTHETIC TECHNICAL VALIDATION — NOT BAF RESULTS
960 generated rows; reduced budgets; two CPU seeds. This checks execution and evidence, not research performance.

# 12 — Focal, Class-Balanced và LDAM–DRW

Chạy CE/balanced, Focal, CB-CE/CB-Focal, DRW, norm-CE, LDAM và LDAM-DRW. LDAM riêng là đối chứng margin loss. Nhóm head chuẩn hóa có baseline norm-CE cùng scale30. Các biến thể của một paper không được tính thành nhiều paper.

Không sửa cấu hình ở notebook này. Muốn đổi cấu hình hãy tạo campaign mới ở11.

In [1]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src" / "baf.py").is_file():
    raise RuntimeError("Mở notebook từ thư mục BAF_project hoặc BAF_project/notebooks.")

CACHE_ROOT = ROOT.parent.parent / ".cache" / "baf-research"
if os.name == "nt":
    CACHE_ROOT = Path(r"D:\UIT\.cache\baf-research")
for name, subdirectory in {
    "PIP_CACHE_DIR": "pip",
    "TORCH_HOME": "torch",
    "XDG_CACHE_HOME": "misc",
    "TEMP": "temp",
    "TMP": "temp",
}.items():
    folder = CACHE_ROOT / subdirectory
    folder.mkdir(parents=True, exist_ok=True)
    os.environ[name] = str(folder)

sys.path.insert(0, str(ROOT / "src"))
import gru

print("Python:", sys.executable)
print("Project:", ROOT)
print("Module:", gru.RESEARCH_VERSION)


Python: /opt/codex/runtimes/codex-primary-runtime/dependencies/python/bin/python3
Project: /workspace/scratch/a3538503433e/work/research11/synthetic_validation
Module: baf-research-20261028-v1


In [2]:
# Điền đường dẫn in ở notebook 11 nếu đã có nhiều campaign.
# Ví dụ: Path(r"D:\UIT\DS317\BAF_project\results\research_campaigns\weighting_research_...")
CAMPAIGN_DIR = None
CAMPAIGN = gru.research_resolve_campaign(ROOT, CAMPAIGN_DIR)
PLAN = gru.research_read(CAMPAIGN / "campaign.json")

print("Campaign:", CAMPAIGN)
print("Mode:", PLAN["mode"])
print("Suites:", PLAN["suites"])
print("Seeds:", PLAN["config"]["seeds"])


Campaign: /workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/research_campaigns/weighting_research_20261028_142640_1f37cc05
Mode: synthetic
Suites: ['losses', 'meta', 'lsr', 'xgboost']
Seeds: [42, 317]


## 1. Chạy training có notebook output thật

        Cell khởi chạy một Python process mới bằng đúng interpreter của kernel. Process thực thi từng cell trong notebook con, lưu stdout/display/error sau từng cell và in log trực tiếp bên dưới. Fit hoàn tất được kiểm hash rồi reuse khi resume. Nếu bị ngắt, không xóa campaign. Cell này có thể chạy lâu.

In [3]:
SUITE = 'losses'
RUN_DIR = None
if SUITE in PLAN["suites"]:
    RUN_DIR = gru.research_launch_suite(CAMPAIGN, SUITE)
else:
    print("SKIPPED: suite không thuộc phạm vi đã chọn ở11:", SUITE)


BEGIN_STAGE 1
END_STAGE 1
BEGIN_STAGE 2
RESEARCH_STUDY losses mode= synthetic device= cpu
END_STAGE 2
BEGIN_STAGE 3
FIT_BEGIN ce_c00_s42 {'method': 'ce', 'lr': 0.03, 'candidate': 0, 'family': 'nn', 'normed': False, 'feature': 'dense'}
EPOCH ce 42 {'epoch': 1, 'loss': 0.661321186617519, 'validation_ap': 0.150611798834493, 'seconds': 0.012285709999559913}
EPOCH ce 42 {'epoch': 2, 'loss': 0.42559970932060415, 'validation_ap': 0.16644238710985876, 'seconds': 0.009244509000382095}
FIT_COMPLETE ce_c00_s42 validation_AP= 0.16644238710985876
FIT_BEGIN balanced_c00_s42 {'method': 'balanced', 'lr': 0.03, 'candidate': 0, 'family': 'nn', 'normed': False, 'feature': 'dense'}
EPOCH balanced 42 {'epoch': 1, 'loss': 0.7114672613947579, 'validation_ap': 0.1607764462986907, 'seconds': 0.009176294999633683}
EPOCH balanced 42 {'epoch': 2, 'loss': 0.6996112418978402, 'validation_ap': 0.16062294040875721, 'seconds': 0.009059780000825413}
FIT_COMPLETE balanced_c00_s42 validation_AP= 0.1607764462986907
FIT_BE

## 2. Xem kết quả validation đã khóa

        Chưa có test metric ở notebook này. Model, configuration và threshold được lưu trước khi mở test ở16.

In [4]:
if RUN_DIR is not None:
    display(gru.research_suite_overview(RUN_DIR))
    print("Notebook training thật:")
    for executed in sorted((RUN_DIR / "attempts").glob("*/training_executed.ipynb")):
        print(executed)


Notebook training thật:
/workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/research_campaigns/weighting_research_20261028_142640_1f37cc05/suites/losses/attempts/20261028_142640_6fbabe6b/training_executed.ipynb


,method,seed,validation_ap,threshold,fit_seconds,family,feature
0,balanced,42,0.160776,inf,0.217787,nn,dense
1,balanced,317,0.251429,0.545188,0.299354,nn,dense
2,cb_ce,42,0.166843,0.567025,0.216907,nn,dense
3,cb_ce,317,0.174976,inf,0.306574,nn,dense
4,cb_focal,42,0.228678,0.518425,0.250427,nn,dense
5,cb_focal,317,0.150746,inf,0.367578,nn,dense
6,ce,42,0.166442,0.113915,1.306487,nn,dense
7,ce,317,0.273211,0.139467,0.294337,nn,dense
8,drw,42,0.195364,0.350916,0.245382,nn,dense
9,drw,317,0.252928,0.431528,0.246198,nn,dense


## 3. Xuất proof trong một lượt

        Không cần Ctrl+S hoặc chạy cell cuối hai lần. Full proof giữ trên máy; review nhỏ hơn để gửi, ghi rõ phần đã bỏ. Notebook `verify.ipynb` được execute thật, kiểm hash/score và không fit lại.

In [5]:
if RUN_DIR is not None:
    PROOF = gru.research_export_proof(RUN_DIR, ROOT)
    for kind, path in PROOF.items():
        print(kind, "→", path)


RESEARCH_FROZEN_SUITE_VERIFIED 18
FULL_PROOF /workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/proofs/losses_15154335_full_proof.zip
PARTIAL_REVIEW /workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/proofs/losses_15154335_review.zip
full_proof → /workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/proofs/losses_15154335_full_proof.zip
review → /workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/proofs/losses_15154335_review.zip
